# Privacy-First AI Interviewer with Gemma 4

**Kaggle Build with Gemma — ML Nashik · Gemma 4 E2B native-audio edition**

This notebook runs the real **Gemma 4 E2B any-to-any** model from Hugging Face on a Kaggle T4. No llama.cpp, no whisper, no separate ASR model: Gemma understands audio natively. Record an answer, and the same model transcribes it, scores it against the interview question, and writes the interviewer's reply. gTTS speaks the reply back.

On a 6 GB laptop GPU this native-audio path does not fit, which is why the local repo (`build-with-gemma/`) splits the job into faster-whisper + llama.cpp + Kokoro. On a cloud T4 (16 GB) the whole pipeline runs inside one model.

## How to run

1. Accept the Gemma license at https://huggingface.co/google/gemma-4-E2B-it (the model is gated).
2. Add a Kaggle secret named `HF_TOKEN` with a read-only Hugging Face token (Account > Settings > Secrets).
3. Run cells top to bottom. **Cell 9 is a self-test** — it transcribes a sample audio file to prove the native-audio path works before you touch the mic. The last cell launches the mic demo.

In [ ]:
%pip install -q torch accelerate soundfile librosa
%pip install -q "transformers>=5.10.1"
%pip install -q gradio gTTS

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

# Gemma 4 is gated. Add a Kaggle secret named HF_TOKEN with a read token.
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

from huggingface_hub import whoami
print('HF user:', whoami()['name'])

In [ ]:
import torch
from transformers import pipeline

MODEL_ID = 'google/gemma-4-E2B-it'

# fp16 because Kaggle's T4 does not support bf16. ~10 GB of weights + audio
# buffers fits inside 16 GB. The official cookbook uses the same task/model.
pipe = pipeline(
    task='any-to-any',
    model=MODEL_ID,
    device_map='auto',
    dtype=torch.float16,
)
print('Loaded', MODEL_ID)

# If VRAM is ever tight, 4-bit instead of fp16:
# from transformers import BitsAndBytesConfig
# pipe = pipeline(task='any-to-any', model=MODEL_ID, device_map='auto',
#                 model_kwargs={'quantization_config': BitsAndBytesConfig(load_in_4bit=True)})

In [ ]:
BANK = [
    {
        'question': 'What is the difference between supervised and unsupervised learning? Give one example of each.',
        'keywords': ['supervised learning uses labeled data', 'unsupervised learning uses unlabeled data', 'classification or regression example', 'clustering or dimensionality reduction example'],
        'explanation': 'Supervised learning trains on labeled data (input-output pairs) and learns a mapping from inputs to outputs. Examples: predicting house prices (regression), spam classification. Unsupervised learning finds patterns in unlabeled data. Examples: customer clustering, PCA.',
    },
    {
        'question': 'Explain the bias-variance tradeoff. What happens when a model is underfitting versus overfitting?',
        'keywords': ['bias is error from oversimplifying', 'variance is error from sensitivity to training data', 'underfitting is high bias low variance', 'overfitting is low bias high variance', 'goal is to minimize total error'],
        'explanation': 'Bias is the error from a model that is too simple; variance is the error from being too sensitive to training noise. Underfitting (high bias) performs poorly on both training and test data. Overfitting (high variance) memorizes training noise and fails on test data. The goal is the complexity that minimizes total error.',
    },
    {
        'question': 'What is cross-validation and why do we use it?',
        'keywords': ['splits data into train and validation folds', 'evaluates the model on held-out data', 'reduces overfitting', 'uses each fold as validation once', 'k-fold'],
        'explanation': 'Cross-validation splits the data into k folds, trains on k-1 of them and validates on the remaining fold, rotating so each fold is the validation set once. It estimates how well the model generalizes to unseen data and helps detect overfitting, using every example for both training and validation at some point.',
    },
]

In [ ]:
import json

Q = chr(34)  # double-quote char, kept in a variable so cells stay readable


def parse_json(text):
    """Lenient JSON parse: strict json.loads first, then a fallback that scans
    for key: value pairs. Mirrors the local repo's retry+regex approach."""
    try:
        return json.loads(text)
    except Exception:
        pass
    result = {}
    for key in ('relevance_score', 'completeness_score', 'clarity_score', 'feedback', 'transcript'):
        needle = Q + key + Q
        i = text.find(needle)
        if i < 0:
            continue
        j = text.find(':', i)
        rest = text[j + 1:].lstrip()
        if rest.startswith(Q):
            k = rest.find(Q, 1)
            val = rest[1:k] if k >= 0 else rest[1:]
        else:
            val = rest.split(',')[0].split('}')[0].strip()
        if key.endswith('_score'):
            try:
                val = int(val)
            except ValueError:
                continue
        result[key] = val
    i = text.find('gaps')
    if i >= 0:
        a = text.find('[', i)
        b = text.find(']', a)
        if a >= 0 and b > a:
            raw = text[a + 1:b]
            result['gaps'] = [g.strip().strip(Q) for g in raw.split(',') if g.strip()]
    return result

In [ ]:
from transformers import GenerationConfig


def transcribe_audio(audio_path):
    """Gemma 4 E2B native ASR: audio in, transcript out. No whisper needed."""
    cfg = GenerationConfig.from_pretrained(MODEL_ID)
    cfg.max_new_tokens = 128
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'text', 'text': 'Transcribe the following speech in English, keeping the candidate wording. Output only the transcript, no newlines. If there is no speech, output [no answer].'},
            {'type': 'audio', 'audio': audio_path},
        ],
    }]
    out = pipe(messages, return_full_text=False, generate_kwargs=dict(generation_config=cfg))[0]['generated_text']
    return out.strip()

In [ ]:
def evaluate_answer(transcript, question, keywords):
    """Text-only eval: transcript + question -> scores as JSON."""
    cfg = GenerationConfig.from_pretrained(MODEL_ID)
    cfg.max_new_tokens = 256
    kw = ', '.join(keywords)
    prompt = ('You are an AI interviewer. Judge the candidate answer for the interview question. '
              'Ignore filler words like um or like. Score each of relevance, completeness, clarity from 1 to 10. '
              'Return ONLY valid JSON with keys: relevance_score, completeness_score, clarity_score, gaps (array of missing key points), feedback (one concrete coaching pointer; if the answer is complete, say so and add nothing). '
              'Question: ' + question + '. Expected key points: ' + kw + '. Candidate answer: ' + transcript)
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': prompt}]}]
    out = pipe(messages, return_full_text=False, generate_kwargs=dict(generation_config=cfg))[0]['generated_text']
    return parse_json(out)


def generate_reply(transcript, feedback, next_question, done):
    """Text-only reply: warm acknowledgement + coaching + next question."""
    cfg = GenerationConfig.from_pretrained(MODEL_ID)
    cfg.max_new_tokens = 256
    if done:
        prompt = ('The interview is over. Thank the candidate warmly and briefly, and say goodbye. Keep it to two sentences.')
    else:
        prompt = ('You are a warm human interviewer. Your name is Mira. Acknowledge what the candidate actually said in one sentence, '
                  'then weave in this coaching pointer if useful: ' + (feedback or 'no note') + '. '
                  'Then ask the next question naturally, keeping the question text exactly: ' + next_question)
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': prompt}]}]
    out = pipe(messages, return_full_text=False, generate_kwargs=dict(generation_config=cfg))[0]['generated_text']
    return out.strip()

In [ ]:
def speak(text, path='/kaggle/working/reply.mp3'):
    """gTTS: interviewer text -> speech. Non-fatal if unavailable."""
    try:
        from gtts import gTTS
        gTTS(text=text, lang='en').save(path)
        return path
    except Exception as exc:
        print('TTS failed:', exc)
        return None

## Self-test (run before the mic demo)

The cell below transcribes a sample audio file from the official Gemma cookbook. If it prints a sentence, the model, the GPU, and the native-audio path are all working.

In [ ]:
sample = 'https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/apps/sample-data/journal1.wav'
print('Transcribing sample audio...')
print(transcribe_audio(sample))

In [ ]:
def run_interview_turn(audio_path, q_index=0):
    """One full interview turn: ASR -> eval -> reply -> TTS."""
    question = BANK[q_index]['question']
    keywords = BANK[q_index]['keywords']

    transcript = transcribe_audio(audio_path)
    result = evaluate_answer(transcript, question, keywords)

    scores = {k: result.get(k) for k in ('relevance_score', 'completeness_score', 'clarity_score')}
    gaps = result.get('gaps', [])
    feedback = result.get('feedback', '')

    if q_index + 1 < len(BANK):
        next_question = BANK[q_index + 1]['question']
        done = False
    else:
        next_question = None
        done = True

    reply = generate_reply(transcript, feedback, next_question, done)
    audio = speak(reply)

    return {
        'transcript': transcript,
        'scores': scores,
        'gaps': gaps,
        'feedback': feedback,
        'reply': reply,
        'reply_audio': audio,
        'next_question': next_question,
        'done': done,
    }

## Mic demo

Run the cell below, pick a question, record your answer, and stop. The notebook will show your transcript, the three scores, a coaching note, the interviewer's reply, and play the spoken reply.

In [ ]:
import gradio as gr


def handle(audio, idx):
    if audio is None:
        return 'No audio recorded.', '', '', '', None
    res = run_interview_turn(audio, q_index=int(idx or 0))
    scores_txt = ' | '.join(f'{k} = {v}' for k, v in res['scores'].items() if v is not None)
    return res['transcript'], scores_txt, res['feedback'], res['reply'], res['reply_audio']


with gr.Blocks(title='AI Interviewer') as demo:
    gr.Markdown('# AI Interviewer — Gemma 4 E2B native audio')
    q = gr.Dropdown(choices=list(range(len(BANK))), value=0, label='Question index')
    mic = gr.Audio(type='filepath', sources=['microphone'], label='Record your answer')
    out_transcript = gr.Textbox(label='Transcript')
    out_scores = gr.Textbox(label='Scores (relevance / completeness / clarity)')
    out_feedback = gr.Textbox(label='Coaching note')
    out_reply = gr.Textbox(label='Interviewer reply')
    out_audio = gr.Audio(label='Spoken reply')
    mic.stop_recording(handle, inputs=[mic, q], outputs=[out_transcript, out_scores, out_feedback, out_reply, out_audio])

demo.launch()